# 20260814_단지_기본정보에서 안양 apartment.csv 추출

## 개요

K-apt 전국 단지 기본정보 엑셀(2026-08-14 추출, 21,687단지)에서 **안양시 동안구**만 골라
운영 파이프라인 계약 CSV `apartments.csv`로 만든다.

- 원천: `../data/Anyang Pyeongchon/20260814_단지_기본정보.xlsx` (시트 1장, 첫 행은 안내문·둘째 행이 헤더)
- 대상: `시군구 == '안양동안구'` 129단지. 만안구 85단지는 제외한다 — `pyeongchon` 지역은
  `src/coolingverse_pipeline/transform.py`가 "안양시 동안구"로 정의하고 격자·단속 인계본도 동안구 범위라,
  만안구를 넣으면 수요(단속) 없는 가장자리 격자에 공급만 붙는다.
- 산출 (계약 11컬럼, `oracle_loader._replace_source_rows` whitelist 기준):
  `region_code, grid_code, kapt_code, name, address, lat, lng, total_parking, is_open, open_count, source`
- 변환 규칙은 부천 셀(`02_Apartment_Preprocessing` [11])을 그대로 따르고, 유휴율만 안양 값
  32.09%(`02` [13])를 쓴다.

흐름: 로드·필터 → 정제 → 지오코딩(카카오, 캐시) → 격자 매핑 → 계약 CSV.


In [ ]:
import re
from pathlib import Path

import pandas as pd

# 경로 - 로컬 클론 기준. Colab이면 '/content/drive/MyDrive/...' 로 바꾼다.
DATA_DIR = Path('../data/Anyang Pyeongchon')
XLSX_PATH = DATA_DIR / '20260814_단지_기본정보.xlsx'
GRID_PATH = DATA_DIR / 'grids_anyang_final.csv'
CONTRACT_DIR = DATA_DIR / 'contract'

REGION_CODE = 'pyeongchon'
TARGET_SIGUNGU = '안양동안구'  # K-apt 표기. '안양시 동안구'가 아니라 '안양동안구'로 들어 있다.

# 1. 로드 - 0행은 K-apt 안내문이므로 1행을 헤더로 읽는다.
df_all = pd.read_excel(XLSX_PATH, header=1)
print(f"전국 단지 {len(df_all):,}건, 컬럼 {df_all.shape[1]}개")

# 2. 안양 동안구 필터
df_anyang = df_all[df_all['시군구'].astype(str).str.contains('안양')]
df_raw = df_all[df_all['시군구'] == TARGET_SIGUNGU].copy().reset_index(drop=True)
print(f"안양시 전체 {len(df_anyang)}건 -> {df_anyang['시군구'].value_counts().to_dict()}")
print(f"동안구 {len(df_raw)}건 채택 (동별: {df_raw['동리'].value_counts().to_dict()})")

# 3. 원천 건전성
assert df_raw['단지코드'].notna().all() and not df_raw['단지코드'].duplicated().any(), '단지코드 누락/중복'
assert df_raw['법정동주소'].notna().all(), '법정동주소 누락'
print(f"도로명주소 결측 {df_raw['도로명주소'].isna().sum()}건 (법정동주소로 대체 가능)")
print(f"총주차대수 0인 단지 {(df_raw['총주차대수'] == 0).sum()}건: "
      f"{df_raw.loc[df_raw['총주차대수'] == 0, '단지명'].tolist()}")


## 정제

부천 셀과 같은 규칙으로 계약 컬럼을 만든다.

- `address`: 법정동주소를 정제해 쓴다. K-apt는 `경기도 안양동안구 호계동 927 호계흥화아파트`처럼
  시(市)가 빠져 있고 번지 뒤에 단지명이 붙으며, 여러 필지가 콤마로 나열되기도 한다.
  첫 필지만 남기고 `안양동안구 -> 안양시 동안구`, 번지까지만 자른다. 도로명주소는 지오코딩 1순위로만 쓴다.
- `total_parking`: `총주차대수` 정수. 0인 3단지(동양월드타워·LD범계마을·HHI브라운빌3차)는 부천과 같이 그대로 둔다(추정하지 않는다).
- `is_open`: 외부인개방여부(지상/지하) 중 하나라도 `허용·개방·Y`면 Y.
- `open_count = round(total_parking * 0.3209)`. 안양 유휴율 산출 근거는 `02_Apartment_Preprocessing` [12,13].


In [ ]:
# 안양시 동안구 유휴율 - 02_Apartment_Preprocessing [13] 과 동일한 산식
APARTMENT_COMMUTE_RATE = 0.788
CAR_SHARE = 105_529 / 259_171  # KOSIS 인구주택총조사(2020) 안양시 통근, 승용차+승합차
IDLE_RATE = APARTMENT_COMMUTE_RATE * CAR_SHARE
print(f"IDLE_RATE = {IDLE_RATE:.4f} ({IDLE_RATE:.2%})")


def clean_jibun_address(address):
    """K-apt 법정동주소를 지오코딩 가능한 지번 주소 한 건으로 만든다."""
    if pd.isna(address) or not str(address).strip():
        return None
    addr = str(address).split(',')[0].strip()             # 첫 필지만
    addr = addr.replace('안양동안구', '안양시 동안구').replace('안양만안구', '안양시 만안구')
    match = re.match(r'^(.+?\s\d+(?:-\d+)?)', addr)          # '... 동 927-3' 까지만
    if match:
        addr = match.group(1)
    return addr.rstrip('-').strip()                        # '374-' 같은 꼬리 하이픈 제거


def clean_road_address(address):
    if pd.isna(address) or not str(address).strip():
        return None
    return str(address).split(',')[0].strip()


def convert_is_open(row):
    ground = str(row['외부인개방여부(지상)']) if pd.notna(row['외부인개방여부(지상)']) else ''
    underground = str(row['외부인개방여부(지하)']) if pd.notna(row['외부인개방여부(지하)']) else ''
    open_keywords = ['허용', '개방', 'Y']
    if any(k in ground for k in open_keywords) or any(k in underground for k in open_keywords):
        return 'Y'
    return 'N'


df_apt = pd.DataFrame({
    'region_code': REGION_CODE,
    'kapt_code': df_raw['단지코드'].astype(str),
    'name': df_raw['단지명'].astype(str).str.strip(),
    'address': df_raw['법정동주소'].map(clean_jibun_address),
    'road_address': df_raw['도로명주소'].map(clean_road_address),   # 지오코딩용, 계약 컬럼 아님
    'total_parking': pd.to_numeric(df_raw['총주차대수'], errors='coerce').fillna(0).astype(int),
    'is_open': df_raw.apply(convert_is_open, axis=1),
})
df_apt['open_count'] = (df_apt['total_parking'] * IDLE_RATE).round().astype(int)
df_apt['source'] = 'K-apt 단지 기본정보(2026-08-14)'

assert df_apt['address'].notna().all()
assert df_apt['address'].str.contains('안양시 동안구').all()

print(f"정제 {len(df_apt)}단지 | 총주차 {df_apt['total_parking'].sum():,}면 -> 유휴 {df_apt['open_count'].sum():,}면")
print(f"is_open: {df_apt['is_open'].value_counts().to_dict()}")
print("\n주소 정제 예시:")
for before, after in zip(df_raw['법정동주소'].head(4), df_apt['address'].head(4)):
    print(f"  {before}\n   -> {after}")

CLEAN_PATH = DATA_DIR / '안양_동안구_아파트_정제.csv'
df_apt.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')
print(f"\n중간 산출 저장: {CLEAN_PATH}")


## 지오코딩 (카카오 로컬 API)

`KAKAO_REST_API_KEY` 환경변수 또는 아래 상수에 키를 넣고 실행한다. 결과는 `geocode_cache_apartments.json`에
캐시하므로 재실행 시 API를 다시 부르지 않는다.

우선순위: ① 도로명주소 → ② 정제 지번주소 → ③ 키워드 검색 `안양 {단지명}` (부천 구제 셀과 동일).
키가 없으면 좌표를 비워 둔 채 지나가고, 다음 셀에서 계약 CSV 저장을 막는다.


In [ ]:
import json
import os
import time

import requests

KAKAO_API_KEY = os.environ.get('KAKAO_REST_API_KEY', 'YOUR_KAKAO_API_KEY')
CACHE_PATH = DATA_DIR / 'geocode_cache_apartments.json'

cache = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}
headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}


def kakao(query, endpoint='address.json'):
    if query in cache:
        return tuple(cache[query]) if cache[query] else None
    res = requests.get(f'https://dapi.kakao.com/v2/local/search/{endpoint}',
                       params={'query': query}, headers=headers, timeout=10)
    res.raise_for_status()
    docs = res.json().get('documents', [])
    found = (float(docs[0]['y']), float(docs[0]['x'])) if docs else None
    cache[query] = list(found) if found else None
    time.sleep(0.03)
    return found


def geocode_row(row):
    for query, endpoint in ((row['road_address'], 'address.json'),
                            (row['address'], 'address.json'),
                            (f"안양 {row['name']}", 'keyword.json')):
        if query:
            found = kakao(query, endpoint)
            if found:
                return found
    return (None, None)


if KAKAO_API_KEY == 'YOUR_KAKAO_API_KEY':
    print('⚠️ 카카오 API 키가 없어 지오코딩을 건너뛴다. lat/lng 는 NaN 으로 남는다.')
    df_apt['lat'], df_apt['lng'] = None, None
else:
    coords = [geocode_row(row) for _, row in df_apt.iterrows()]
    df_apt['lat'] = [c[0] for c in coords]
    df_apt['lng'] = [c[1] for c in coords]
    CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding='utf-8')

matched = df_apt['lat'].notna().sum()
print(f"좌표 매칭 {matched}/{len(df_apt)} ({matched / len(df_apt):.2%})")
if matched < len(df_apt):
    print('미매칭(최대 20건):', df_apt.loc[df_apt['lat'].isna(), 'name'].head(20).tolist())


## 격자 매핑 → 계약 CSV

`grids_anyang_final.csv`의 격자 중심점에 KD-Tree로 최근접 매핑하고 자연키 `grid_code`를 붙인다.
`transform.map_nearest_grid`와 같이 약 3 km(0.03°) 초과는 타 지역 좌표로 보고 매핑하지 않는다.

품질 게이트(`quality.validate`)가 아파트 좌표 매칭률 100%를 요구하므로, 미매칭이 있으면 계약 CSV를 쓰지 않는다.


In [ ]:
import numpy as np
from scipy.spatial import cKDTree

df_grid = pd.read_csv(GRID_PATH, encoding='utf-8-sig', dtype={'grid_code': str})
tree = cKDTree(df_grid[['center_lat', 'center_lng']].to_numpy(float))

df_apt['grid_code'] = None
has_coord = df_apt['lat'].notna() & df_apt['lng'].notna()
if has_coord.any():
    distances, indices = tree.query(df_apt.loc[has_coord, ['lat', 'lng']].to_numpy(float), k=1)
    accepted = np.asarray(distances) <= 0.03
    codes = df_grid.iloc[np.asarray(indices)]['grid_code'].to_numpy(object)
    df_apt.loc[df_apt.index[has_coord][accepted], 'grid_code'] = codes[accepted]

# 격자 bbox 밖(안양 동안구 밖) 좌표는 잘못 지오코딩된 것이므로 확인한다.
bbox = dict(lat=(df_grid['min_lat'].min(), df_grid['max_lat'].max()),
            lng=(df_grid['min_lng'].min(), df_grid['max_lng'].max()))
outside = has_coord & ~(df_apt['lat'].between(*bbox['lat']) & df_apt['lng'].between(*bbox['lng']))
if outside.any():
    print('⚠️ 격자 범위 밖 좌표 (주소 확인 필요):')
    print(df_apt.loc[outside, ['name', 'address', 'lat', 'lng']].to_string())

CONTRACT_COLUMNS = ['region_code', 'grid_code', 'kapt_code', 'name', 'address', 'lat', 'lng',
                    'total_parking', 'is_open', 'open_count', 'source']
df_contract = df_apt[CONTRACT_COLUMNS]

grid_rate = df_contract['grid_code'].notna().mean()
print(f"격자 매칭 {df_contract['grid_code'].notna().sum()}/{len(df_contract)} ({grid_rate:.2%}), "
      f"공급 격자 {df_contract['grid_code'].nunique()}개")

if grid_rate == 1.0:
    CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
    out = CONTRACT_DIR / 'apartments.csv'
    df_contract.to_csv(out, index=False, encoding='utf-8')
    print(f"🎉 계약 CSV 저장: {out} ({len(df_contract)}행, open_count 합계 {df_contract['open_count'].sum():,})")
else:
    print('❌ 매칭률 100% 미만 - 계약 CSV를 쓰지 않는다. 지오코딩 셀을 키와 함께 다시 실행하거나 미매칭 주소를 손본다.')
df_contract.head()
